In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Silver Payments Validation").getOrCreate()



c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
bronze_payments = spark.read.parquet("../data/bronze/payments")
silver_payments = spark.read.parquet("../data/silver/payments")

In [5]:
from pyspark.sql.functions import col, count, when

In [4]:
print("Bronze payments:", bronze_payments.count())
print("Silver Payments:", silver_payments.count())


Bronze payments: 103886
Silver Payments: 103886


In [6]:
silver_payments.select(
    [
        count(when(col(c).isNull(),c)).alias(c)
        for c in silver_payments.columns
    ]
).show(vertical=True)

-RECORD 0-------------------
 order_id             | 0   
 payment_sequential   | 0   
 payment_type         | 0   
 payment_installments | 0   
 payment_value        | 0   



In [8]:
duplicate_silver_payments = silver_payments.groupBy("order_id","payment_sequential").count().filter(col("count")>1)
duplicate_silver_payments.show()
print(f"Duplicate silver payments:", duplicate_silver_payments.count())

+--------+------------------+-----+
|order_id|payment_sequential|count|
+--------+------------------+-----+
+--------+------------------+-----+

Duplicate silver payments: 0


In [ ]:
invalid_payment_value = silver_payments.filter(col("payment_value") < 0)
invalid_payment_value.show()
invalid_payment_value.count()

+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
+--------+------------------+------------+--------------------+-------------+



0

In [14]:
invalid_installments = silver_payments.filter(col("payment_installments") < 1)
invalid_installments.count()
print("Invalid Installments:",invalid_installments.count())

Invalid Installments: 2


In [12]:
silver_payments.filter(
    col("payment_installments") == 0
).select(
    "payment_type",
    "payment_value"
).show()

+------------+-------------+
|payment_type|payment_value|
+------------+-------------+
| credit_card|        58.69|
| credit_card|       129.94|
+------------+-------------+



Payment Installment Validation:

Rule:
payment_installments should normally be >= 1.

Result:
2 records were found with payment_installments = 0.

Action:
Records were retained to preserve source financial data.
Records retained and flagged for review

In [15]:
valid_payment_types = [
    "boleto",
    "credit_card",
    "debit_card",
    "voucher",
    "not_defined"
]
invalid_payment_type = silver_payments.filter(~col("payment_type").isin(valid_payment_types))
print("Invalid payment types:", invalid_payment_type.count())

Invalid payment types: 0
